# Unidad 2: Aprendizaje Automático 
## 🗺️ Búsqueda en Grilla Multidimensional — GridSearchCV
### Inteligencia Artificial - Lic. en Sistemas - FCAD/UNER

---

## 🎯 ¿Qué vamos a aprender?

En el notebook anterior exploramos un solo hiperparámetro (`n_estimators`). Ahora vamos a **explorar múltiples hiperparámetros simultáneamente**, formando una grilla multidimensional.

Al finalizar, vas a poder:
- ✅ Explorar **combinaciones de múltiples hiperparámetros** con GridSearchCV
- ✅ Entender el parámetro `max_features` de Random Forest
- ✅ Optimizar usando **Precision** como métrica de evaluación
- ✅ Visualizar el espacio de hiperparámetros como un **heatmap**
- ✅ Justificar la elección de métrica según el contexto del problema

---

## 🧠 Conceptos Clave

### 🗺️ Búsqueda Multidimensional

Cuando definimos un `param_grid` con **múltiples parámetros**, GridSearchCV evalúa **todas las combinaciones posibles**:

```
n_estimators:  [10,  25,  50,  70,  72,  75,  78,  80,  100, 120]
max_features:  ['sqrt', 'log2']

→ 10 × 2 = 20 combinaciones × cv=10 folds = 200 modelos entrenados
```

Este espacio de búsqueda se puede visualizar como una **tabla 2D** donde cada celda contiene el score de esa combinación.

### 🌲 El Parámetro `max_features`

En cada división de un árbol, Random Forest evalúa **sólo un subconjunto** de las features disponibles. Esto reduce la correlación entre árboles.

| `max_features` | Subconjunto evaluado | Uso recomendado |
|----------------|---------------------|------------------|
| `'sqrt'` | √n_features | Clasificación ✅ |
| `'log2'` | log₂(n_features) | Alternativa válida |
| `None` / `1.0` | Todas las features | Como un Bagging solo |
| Entero `k` | k features | Control manual |

> Para **30 features** (Breast Cancer): `'sqrt'` → 5-6 features por split; `'log2'` → 5 features.

### 🎯 ¿Por qué optimizar Precision en este caso?

Dependiendo del contexto clínico, podríamos priorizar:
- 🔴 **Recall alto**: No queremos perder casos malignos → Minimizar **Falsos Negativos**
- 🔵 **Precision alta**: No queremos alarmar a pacientes sanos → Minimizar **Falsos Positivos**

En este notebook elegimos `scoring='precision'` para practicar la búsqueda orientada a minimizar falsos positivos.

> 📌 **Referencias:**
> - Scikit-learn: [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
> - Scikit-learn: [RandomForestClassifier — max_features](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
> - Géron, A. (2019). *Hands-On Machine Learning*, Cap. 7 (Ensemble Learning). O'Reilly.

---

## 📦 Paso 1: Importar las Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

print('✅ Librerías importadas correctamente!')

## 📂 Paso 2: Cargar el Dataset

In [ ]:
# 📥 Cargar el dataset
cancer_data = load_breast_cancer()
df = pd.DataFrame(cancer_data['data'], columns=cancer_data['feature_names'])
df['target'] = cancer_data['target']

X = df[cancer_data.feature_names].values
y = df['target'].values

print(f'📐 Dataset: {X.shape[0]} muestras × {X.shape[1]} features')
print(f'🏷️  Clases: {list(cancer_data.target_names)}')

## 🗺️ Paso 3: Definir la Grilla Multidimensional

Ahora definimos **dos hiperparámetros** simultáneamente. La grilla tendrá **10 × 2 = 20** combinaciones.

In [ ]:
# 🗺️ Grilla de hiperparámetros
param_grid = {
    'n_estimators': [10, 25, 50, 70, 72, 75, 78, 80, 100, 120],
    'max_features': ['sqrt', 'log2']
}

n_combis = len(param_grid['n_estimators']) * len(param_grid['max_features'])
cv_folds = 10

print('🗺️  Espacio de búsqueda:')
print(f'   n_estimators : {param_grid["n_estimators"]}')
print(f'   max_features : {param_grid["max_features"]}')
print(f'\n📊 Combinaciones: {n_combis}')
print(f'🔄 Folds CV:      {cv_folds}')
print(f'🤖 Modelos totales a entrenar: {n_combis * cv_folds}')

## 🚀 Paso 4: Ejecutar GridSearchCV

Usamos `scoring='precision'` y `cv=10` (10-Fold Cross-Validation) para una estimación robusta.

In [ ]:
# 🤖 Modelo base
rf = RandomForestClassifier(random_state=150)

# 🔍 GridSearch con 10-Fold CV, optimizando Precision
gs = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring='precision',
    cv=cv_folds,
    verbose=1
)

print('🚀 Iniciando búsqueda en grilla multidimensional...')
gs.fit(X, y)
print('\n✅ Búsqueda completada!')

## 🏆 Paso 5: Resultados y Mejores Parámetros

In [ ]:
print('=' * 55)
print('🏆 MEJORES HIPERPARÁMETROS (optimizando Precision)')
print('=' * 55)
for k, v in gs.best_params_.items():
    print(f'  {k}: {v}')
print(f'\n  📊 Mejor Precision (CV-10): {gs.best_score_:.4f}')

## 🔥 Paso 6: Visualizar el Espacio de Hiperparámetros — Heatmap

Un **heatmap** es ideal para visualizar espacios de búsqueda bidimensionales.

In [ ]:
# 🔥 Construir la tabla de resultados para el heatmap
results = gs.cv_results_
n_est_vals  = param_grid['n_estimators']
max_feat_vals = param_grid['max_features']

# Reorganizar scores en una matriz 2D
scores_matrix = np.array(results['mean_test_score']).reshape(
    len(n_est_vals), len(max_feat_vals)
)

scores_df = pd.DataFrame(
    scores_matrix,
    index=n_est_vals,
    columns=max_feat_vals
)

fig, ax = plt.subplots(figsize=(7, 7))
sns.heatmap(
    scores_df,
    annot=True, fmt='.3f',
    cmap='YlGn',
    linewidths=0.5,
    ax=ax
)
ax.set_title('Precision (mean CV-10) por combinación de hiperparámetros', fontsize=13)
ax.set_xlabel('max_features')
ax.set_ylabel('n_estimators')
plt.tight_layout()
plt.show()

print(f'\n💡 La celda más oscura (verde más intenso) es la mejor combinación.')
print(f'   Mejor: n_estimators={gs.best_params_["n_estimators"]}, max_features="{gs.best_params_["max_features"]}"')

## 📋 Paso 7: Tabla de Resultados Completa

In [ ]:
# 📋 DataFrame con todos los resultados
tabla = pd.DataFrame({
    'n_estimators': results['param_n_estimators'],
    'max_features': results['param_max_features'],
    'Precision_mean': results['mean_test_score'].round(4),
    'Precision_std':  results['std_test_score'].round(4),
    'rank': results['rank_test_score']
}).sort_values('rank')

print('📋 Todos los resultados (ordenados por rank):')
tabla

## 🏁 Conclusiones

En este notebook aprendimos:

1. 🗺️ **GridSearchCV multidimensional** explora todas las combinaciones del producto cartesiano de los valores definidos en `param_grid`.
2. 🌲 `max_features` controla cuántas features evalúa cada árbol en cada split — `'sqrt'` es el valor estándar para clasificación.
3. 🔥 Un **heatmap** permite visualizar intuitivamente qué combinaciones funcionan mejor en un espacio 2D.
4. 🎯 La elección de `scoring` cambia cuál combinación se considera "la mejor": **Precision**, **Recall** y **F1** priorizan aspectos distintos del rendimiento.
5. ⚡ A mayor espacio de búsqueda + mayor `cv`, mayor costo computacional → considerar `RandomizedSearchCV` para espacios grandes.

### ➡️ Próximo notebook: Curva de Codo para determinar el n_estimators óptimo

---

## 📚 Referencias

- Scikit-learn: [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
- Scikit-learn: [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
- Géron, A. (2019). *Hands-On Machine Learning*, Cap. 7. O'Reilly.
- Bergstra, J., & Bengio, Y. (2012). [Random Search for Hyper-Parameter Optimization](https://www.jmlr.org/papers/v13/bergstra12a.html). *JMLR*, 13, 281–305.